# Bug Report Triage Agent: Design of Autonomous and Semi-Autonomous Agentic Workflows

**Udacity Master's in AI, Capstone Project**

## Project Summary

*To be written at the end (4 to 6 sentences), after the system is built and tested.*

## Notebook Contents

1. Setup and data loading
2. State and output schemas
3. Duplicate search tool (TF-IDF)
4. Agents (Router, Duplicate-finder, Triage)
5. Graph wiring and safeguards
6. Test runs and evaluation
7. Failure case and prompt change (before and after)

In [1]:
import os
import pandas as pd
import re

from typing import Literal, TypedDict, Optional
from pydantic import BaseModel, Field, ValidationError
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from collections import Counter

from dotenv import load_dotenv

## 1. Setup and Data Loading

This section loads the two files created in `data_prep.ipynb`:

- `data/index.csv`: 500 bugs used as the search index for duplicate detection.
- `data/test.csv`: 21 test reports (15 sampled, 3 duplicates, 3 edge cases) with the expected outcome of each one.

Files are loaded with `keep_default_na=False` so empty labels stay as empty strings instead of `NaN`. Checks: no missing values, and the `case_type` and `expected_outcome` counts match the plan.

In [2]:
load_dotenv()

True

In [3]:
bugs_df = pd.read_csv('data/index.csv', keep_default_na=False)
bugs_test_df = pd.read_csv('data/test.csv', keep_default_na=False)

In [4]:
bugs_df.head()

,bug_id,product_name,short_description,long_description,component_name,severity
0,PLATFORM-311936,PLATFORM,[Browser][regression] swt-xulrunner-win32 fail...,+++ This bug was initially created as a clone ...,SWT,high
1,Z_ARCHIVED-109187,Z_ARCHIVED,Adding feature description for branding plugins,Each feature has a branding plugin that provid...,TPTP,normal
2,Z_ARCHIVED-300514,Z_ARCHIVED,Statistcal model loader adds CounterDescriptor...,This is not a functional defect but rather a p...,TPTP,high
3,PLATFORM-143992,PLATFORM,Deprecate the org.eclipse.core.commands.util p...,RC5\n\norg.eclipse.core.commands.util is marke...,UI,normal
4,Z_ARCHIVED-272048,Z_ARCHIVED,After upgrading to Java 1.6.0_13 profiling app...,Build ID: M20090211-1700\n\nSteps To Reproduce...,TPTP,high


In [5]:
bugs_test_df.head()

,bug_id,product_name,short_description,long_description,component_name,severity,case_type,expected_original_id,expected_outcome
0,GMF-TOOLING-144903,GMF-TOOLING,.log file indicates that help documentation co...,!ENTRY org.eclipse.help.base 4 4 2006-05-22 16...,Core,high,sampled,,auto_triaged
1,MYLYN-277040,MYLYN,errors in log after opening discovery wizard,I get hundreds of exceptions after opening the...,UI,high,sampled,,auto_triaged
2,PLATFORM-34879,PLATFORM,Calling ScrollBar.setVisible() when a text wid...,This happens even if the scrollbar visibility ...,SWT,high,sampled,,auto_triaged
3,JDT-424138,JDT,[1.8][dom ast] Bad source ranges for qualified...,BETA_JAVA8\n\nBad source ranges for qualified ...,Core,high,sampled,,auto_triaged
4,Z_ARCHIVED-199762,Z_ARCHIVED,[UX][PII] Coverage Statistics view shows incor...,When open Coverage Statistics view with profil...,TPTP,high,sampled,,auto_triaged


In [6]:
print(bugs_df.shape, bugs_test_df.shape)

(500, 6) (21, 9)


In [7]:
bugs_test_df['case_type'].value_counts()

case_type
sampled         15
duplicate        3
vague            1
question         1
out_of_scope     1
Name: count, dtype: int64

In [8]:
print(bugs_df.isna().sum().sum(), bugs_test_df.isna().sum().sum())
print(bugs_test_df['expected_outcome'].value_counts())

0 0
expected_outcome
auto_triaged          15
duplicate              3
rejected               2
needs_human_review     1
Name: count, dtype: int64


## 2. State and Output Schemas

Each agent must return a fixed structure, not free text. I define one Pydantic model per agent:

- `RouterOutput`: route (`bug`, `question`, `out_of_scope`) and a reason.
- `DuplicateOutput`: is_duplicate, matched bug id, and a reason.
- `TriageOutput`: severity, component (top 5 plus `other`), confidence (0 to 1), and a reason.

Fixed choices use `Literal`, so an invalid value is rejected instead of passing silently. `TriageState` is the shared state that moves through the graph for one report.

Test: one valid example is accepted and one invalid example fails validation.

In [9]:
class RouterOutput(BaseModel):
    route: Literal["bug", "question", "out_of_scope"]
    reason: str

class DuplicateOutput(BaseModel):
    is_duplicate: bool
    matched_bug_id: str = ""   # empty if no duplicate
    reason: str

class TriageOutput(BaseModel):
    severity: Literal["low", "normal", "high"]
    component: Literal["UI", "Core", "SWT", "TPTP", "Releng", "Other"]
    confidence: float = Field(ge=0, le=1)
    reason: str

class TriageState(TypedDict):
    report: dict
    router: Optional[dict]
    candidates: list
    duplicate: Optional[dict]
    triage: Optional[dict]
    final_decision: str
    review_reason: str
    step_count: int
    log: list

In [10]:
print(TriageOutput(severity="high", component="UI", confidence=0.8, reason="test"))

try:
    TriageOutput(severity="urgent", component="Debug", confidence=1.5, reason="test")
except ValidationError as e:
    print("Failed as expected:", e.error_count(), "errors")

severity='high' component='UI' confidence=0.8 reason='test'
Failed as expected: 3 errors


In [11]:
print(bugs_df['component_name'].unique())

<StringArray>
['SWT', 'TPTP', 'UI', 'Core', 'Releng']
Length: 5, dtype: str


## 3. Duplicate Search Tool (TF-IDF)

The Duplicate-finder needs a way to find similar bugs in the index. I build a TF-IDF search tool over the `short_description` field only, because many long descriptions are auto-generated and look alike.

- The index is built once from `data/index.csv` (500 bugs).
- Given a new report title, the tool returns the top 3 most similar bugs with their ids, titles and similarity scores.
- The top similarity score is also used later as a measurable signal in the confidence decision.
- The tool is a LangChain `@tool`, but the Duplicate-finder node calls it directly. The search must always run, so letting the LLM decide whether to call it would add risk and no value.

Test: the tool is tested alone first, using the 3 hand-written duplicate reports. The expected original should appear in the top 3.

In [12]:
# Build the index once, on short_description only
vectorizer = TfidfVectorizer(stop_words="english", lowercase=True)
index_matrix = vectorizer.fit_transform(bugs_df["short_description"])

@tool
def search_similar_bugs(title: str, top_k: int = 3) -> list:
    """Find the most similar bugs in the index for a new report title."""
    query_vec = vectorizer.transform([title])
    scores = cosine_similarity(query_vec, index_matrix).ravel()
    top_idx = scores.argsort()[::-1][:top_k]
    return [
        {
            "bug_id": str(bugs_df.iloc[i]["bug_id"]),
            "short_description": bugs_df.iloc[i]["short_description"],
            "score": round(float(scores[i]), 3),
        }
        for i in top_idx
    ]

In [13]:
dups = bugs_test_df[bugs_test_df["case_type"] == "duplicate"]
for _, row in dups.iterrows():
    results = search_similar_bugs.invoke({"title": row["short_description"]})
    found = [r["bug_id"] for r in results]
    print(row["short_description"])
    print("expected:", row["expected_original_id"], "| top 3:", found)
    print(results, "\n")

Text selection drawn incorrectly when selecting backwards
expected: PLATFORM-31385 | top 3: ['PLATFORM-31385', 'NATTABLE-520258', 'PLATFORM-158175']
[{'bug_id': 'PLATFORM-31385', 'short_description': 'Redraw problems: selecting text backwards', 'score': 0.601}, {'bug_id': 'NATTABLE-520258', 'short_description': 'Column Group is not drawn', 'score': 0.295}, {'bug_id': 'PLATFORM-158175', 'short_description': 'Java core [WM_BUTTONDOWN] when selecting a tree in Vista', 'score': 0.203}] 

JavaDoc code completion missing for methods in the same class
expected: JDT-18719 | top 3: ['JDT-18719', 'PLATFORM-46915', 'JDT-29889']
[{'bug_id': 'JDT-18719', 'short_description': 'No code complete in JavaDoc for methods in the same type [javadoc]', 'score': 0.646}, {'bug_id': 'PLATFORM-46915', 'short_description': 'Javadoc for IPerspectiveListener', 'score': 0.337}, {'bug_id': 'JDT-29889', 'short_description': 'Typecomment template ignores variables in Javadoc [code manipulation]', 'score': 0.302}] 

Er

## 4. Agents (Router, Duplicate-finder, Triage)

The system has three agents, all using the same model (`temperature=0`) with structured output:

- **Router**: decides if a report is a `bug`, a `question`, or `out_of_scope`. Only bugs continue.
- **Duplicate-finder**: calls the TF-IDF tool directly, then the LLM judges whether the top candidates are real duplicates.
- **Triage**: assigns severity, component and confidence. The review threshold is decided later, from the real results.

Each node writes its result and a log entry to the state, and increases `step_count`. If an API call fails, the node sends the report to human review instead of crashing.

## 4a. LLM Setup

All three agents use the same model with `temperature=0`, so the same report gives the same answer. The API key is read from the `.env` file and is never written in the notebook.

- Model: Claude Haiku 4.5 (`claude-haiku-4-5-20251001`) through the Anthropic API, using `langchain_anthropic.ChatAnthropic`.
- `temperature=0` for repeatable results.
- Each agent uses structured output, so the answer must match its Pydantic model.

Test: one simple call to check that the key works.

In [14]:
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,
    max_tokens=1000,
)

print(llm.invoke("Reply with the word: ready").content)

ready


## 4b. Router Agent

The Router is the first node. It reads the report and chooses one route:

- `bug`: continues to the Duplicate-finder.
- `question` or `out_of_scope`: the graph ends with the decision `rejected`.

It uses structured output (`RouterOutput`). If the API call fails, the report goes to human review instead of crashing. Each call adds one log entry and increases `step_count`.

The prompt contains general rules only. No test report is written into it.

Test: run the Router alone on all 21 test reports. Only the question and out_of_scope cases should be rejected.

In [15]:
ROUTER_PROMPT = """You are the Router of a bug triage system for the Eclipse project.
Classify the incoming text into exactly one route:
- bug: describes a problem, error, crash, or wrong behavior in Eclipse or an Eclipse-based tool, or asks for a change to it. Choose bug even if the details are few. Missing details are handled later.
- question: asks how to do something or asks for help or usage information.
- out_of_scope: is not about Eclipse or its tools.
Judge only the text you receive. Give a short reason."""

router_llm = llm.with_structured_output(RouterOutput)

def router_node(state: TriageState) -> dict:
    report = state["report"]
    text = (
        f"Title: {report['short_description']}\n"
        f"Description: {str(report.get('long_description', ''))[:1500]}"
    )
    try:
        result = router_llm.invoke([("system", ROUTER_PROMPT), ("human", text)])
        update = {"router": result.model_dump()}
        if result.route != "bug":
            update["final_decision"] = "rejected"
        entry = f"router -> {result.route}: {result.reason}"
    except Exception as e:
        update = {
            "router": None,
            "final_decision": "needs_human_review",
            "review_reason": f"router error: {e}",
        }
        entry = f"router ERROR: {e}"
    update["step_count"] = state["step_count"] + 1
    update["log"] = state["log"] + [entry]
    return update

In [16]:
for _, row in bugs_test_df.iterrows():
    state = {
        "report": row.to_dict(), "router": None, "candidates": [],
        "duplicate": None, "triage": None, "final_decision": "",
        "review_reason": "", "step_count": 0, "log": [],
    }
    out = router_node(state)
    print(row["case_type"], "|", out["router"]["route"] if out["router"] else "ERROR",
          "|", row["short_description"][:45])

sampled | bug | .log file indicates that help documentation c
sampled | bug | errors in log after opening discovery wizard
sampled | bug | Calling ScrollBar.setVisible() when a text wi
sampled | bug | [1.8][dom ast] Bad source ranges for qualifie
sampled | bug | [UX][PII] Coverage Statistics view shows inco
sampled | bug | SourceFile class from org.eclipse.releng.tool
sampled | bug | Enum constructor: should have private method 
sampled | bug | TVT35:TCT412: JPN: missing mnemonic key (Star
sampled | bug | [cheatsheet] Inconsistent wording in RCP chea
sampled | bug | Remove Y-Builds from downloads and updates
sampled | bug | infinite loop during attempted code assist
sampled | bug | Installation guide refers to the wrong file n
sampled | bug | SWT.Settings - no notification on Vista
sampled | bug | provide Keyboard Shortcuts for the analysis a
sampled | bug | Deprecate unused IBM-specific constants in or
duplicate | bug | Text selection drawn incorrectly when selecti
duplicate | bug | J

## 4c. Duplicate-finder Agent

The Duplicate-finder always calls the TF-IDF search tool first, then asks the LLM to judge the top 3 candidates.

- The search result is saved in `candidates` in the state.
- The LLM returns `DuplicateOutput`: is_duplicate, matched bug id and a reason.
- The LLM must choose only from the 3 candidate ids, or none. This is a guard against invented ids.
- The similarity score is shown to the LLM as a hint only. A high score alone is not enough, because the vague report scored 0.560 against an unrelated bug.
- If the LLM says duplicate, the decision is `duplicate` and the graph ends. If not, the report goes to Triage.
- If the API call fails, the report goes to human review.

Test: run the node alone on all 21 test reports. The 3 duplicate reports should match their originals, and the others should not.

In [17]:
DUP_PROMPT = """You are the Duplicate-finder of a bug triage system for the Eclipse project.
You get a new bug report and up to 3 candidate bugs that already exist.
Decide if the new report describes the SAME problem as one candidate.
Rules:
- Duplicate means the same underlying problem, even if the wording is different.
- Sharing a few general words (for example "does not work" or "error") is NOT enough.
- A similar topic or the same component is NOT enough. The problem itself must match.
- The similarity score is only a hint. A high score can still be a wrong match.
- If the new report is too vague to compare, answer not a duplicate.
- matched_bug_id must be exactly one of the candidate ids, or an empty string if there is no duplicate.
Give a short reason."""

dup_llm = llm.with_structured_output(DuplicateOutput)

def duplicate_node(state: TriageState) -> dict:
    report = state["report"]
    try:
        candidates = search_similar_bugs.invoke({"title": report["short_description"]})
        cand_text = "\n".join(
            f"- id: {c['bug_id']} | score: {c['score']} | title: {c['short_description']}"
            for c in candidates
        )
        text = (
            f"New report title: {report['short_description']}\n"
            f"New report description: {str(report.get('long_description', ''))[:1500]}\n\n"
            f"Candidates:\n{cand_text}"
        )
        result = dup_llm.invoke([("system", DUP_PROMPT), ("human", text)])
        valid_ids = {c["bug_id"] for c in candidates}
        if result.is_duplicate and result.matched_bug_id not in valid_ids:
            raise ValueError(f"LLM returned unknown id: {result.matched_bug_id}")
        update = {"candidates": candidates, "duplicate": result.model_dump()}
        if result.is_duplicate:
            update["final_decision"] = "duplicate"
        entry = (f"duplicate -> is_duplicate={result.is_duplicate} "
                 f"id={result.matched_bug_id or '-'}: {result.reason}")
    except Exception as e:
        update = {
            "duplicate": None,
            "final_decision": "needs_human_review",
            "review_reason": f"duplicate-finder error: {e}",
        }
        entry = f"duplicate ERROR: {e}"
    update["step_count"] = state["step_count"] + 1
    update["log"] = state["log"] + [entry]
    return update

In [18]:
for _, row in bugs_test_df.iterrows():
    state = {
        "report": row.to_dict(), "router": None, "candidates": [],
        "duplicate": None, "triage": None, "final_decision": "",
        "review_reason": "", "step_count": 0, "log": [],
    }
    out = duplicate_node(state)
    d = out["duplicate"]
    print(row["case_type"], "|",
          (d["is_duplicate"], d["matched_bug_id"]) if d else "ERROR",
          "| expected:", row["expected_original_id"] or "-")

sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
sampled | (False, '') | expected: -
duplicate | (True, 'PLATFORM-31385') | expected: PLATFORM-31385
duplicate | (True, 'JDT-18719') | expected: JDT-18719
duplicate | (True, 'XTEND-485032') | expected: XTEND-485032
vague | (False, '') | expected: -
question | (False, '') | expected: -
out_of_scope | (False, '') | expected: -


## 4d. Triage Agent

The Triage agent reads a report that is not a duplicate and returns `TriageOutput`: severity (`low`, `normal`, `high`), component (top 5 plus `Other`) and a confidence from 0 to 1.

- The prompt gives general definitions of each severity and component. No test report is written in it.
- The prompt tells the model to give lower confidence when the report is vague or has few details.
- This step stores the LLM's own confidence only. How it is combined with the similarity signal and the review threshold is decided after I see the real numbers.
- If the API call fails, the report goes to human review.

Test: run the node alone on the 15 sampled reports and the vague report. Compare with the true labels. Baseline on this test set: 33% (always `normal`). The real data has about 77% normal.

In [19]:
TRIAGE_PROMPT_V1 = """You are the Triage agent of a bug triage system for the Eclipse project.
Assign a severity and a component to the bug report.

Severity:
- low: cosmetic problem, typo, wording, small inconvenience, or a minor enhancement request.
- normal: a real problem with a workaround, or a normal-impact defect. Use this when unsure between low and high.
- high: crash, hang, data loss, security problem, or a major function that is broken with no workaround.

Component (choose exactly one):
- UI: Eclipse user interface, views, editors, wizards, dialogs.
- Core: core runtime, resources, build, language tooling.
- SWT: the SWT widget toolkit and native platform behavior.
- TPTP: testing and profiling tools.
- Releng: release engineering, builds, downloads, installation and update.
- Other: none of the above fits.

Confidence is a number from 0 to 1 for your own answer. Give a LOW confidence when the report is vague, has very few details, or fits several components. Give a high confidence only when the text clearly supports your answer.
Give a short reason."""

triage_llm = llm.with_structured_output(TriageOutput)

## 4e. Failure Case: TPTP Component (before and after)

**Run 1 (before).** The Triage agent got 0 of 5 TPTP reports right. It never predicted TPTP and chose UI, Releng or Other instead. The cause is probably my short TPTP description in the prompt ("testing and profiling tools").

**Fix.** I rewrite the TPTP description using what TPTP bugs look like in the 500-bug search index. No test report is used to write it.

**Rule for this section.** Run 1 stays in the notebook with its output. The new prompt is run 2, on the same reports. I compare both runs.

In [20]:
tptp = bugs_df[bugs_df["component_name"] == "TPTP"]
print("TPTP bugs in index:", len(tptp))
print(tptp["product_name"].value_counts().head(10))
print()
for t in tptp["short_description"].sample(min(20, len(tptp)), random_state=42):
    print("-", t)

TPTP bugs in index: 55
product_name
Z_ARCHIVED    55
Name: count, dtype: int64

- [Webdoc] 4.1 Installation Guide - fix doc reference in new tech AC
- Add serialVersionUIDs for classes that implement java.io.Serializable.
- Eclipse crash after opening profiling dialog
- Invalid template string
- Bug with FileTransferAgent
- Template XML file isn't correctly queried
- [Webdoc] [Readme] Manual Test View requires a specific vendor/version of Web browser to be installed/configured.
- TPTP test results are not reported correctly.
- Point Feature:  Documentation work for [93212] Launch UI for more agents incl. JVMTI in the summary
- General maintenance bug for TPTP 4.5.2.1
- AC Local Port preference should be public
- Some small typos in arch_tptp_platform.html
- NPE is thrown by the new AC in ConnectionImpl.addDataListener() when client doesn't register a data processor with the console
- java.lang.NoClassDefFoundError for TestFilterExists.testsuite on AS400 platform
- org.eclipse.tptp.trac

### 4e.1 What TPTP bugs look like in the index

I look only at the 55 TPTP bugs in the search index, never at the test reports. Two checks:

- Which words repeat across all 55 titles (the first look used only a sample of 20).
- Whether the product name `Z_ARCHIVED` is specific to TPTP. If it is, it is a dataset artifact and I do not use it in the prompt.

In [21]:
words = Counter(
    w for t in tptp["short_description"]
    for w in re.findall(r"[a-z]+", t.lower()) if len(w) > 3
)
print(words.most_common(25))

print((bugs_df["product_name"] == "Z_ARCHIVED").groupby(bugs_df["component_name"]).sum())

[('tptp', 11), ('java', 5), ('profiling', 5), ('test', 5), ('platform', 4), ('copyright', 4), ('eclipse', 4), ('readme', 4), ('plugins', 3), ('after', 3), ('trace', 3), ('missing', 3), ('results', 3), ('view', 3), ('editor', 3), ('project', 3), ('with', 3), ('when', 3), ('webdoc', 3), ('build', 3), ('feature', 2), ('model', 2), ('attribute', 2), ('errors', 2), ('plugin', 2)]
component_name
Core       0
Releng     0
SWT        0
TPTP      55
UI         0
Name: product_name, dtype: int64


In [22]:
print(bugs_test_df.groupby("component_name")["product_name"].apply(lambda s: (s == "Z_ARCHIVED").sum()))

component_name
          0
Core      0
Releng    0
SWT       0
TPTP      5
UI        0
Name: product_name, dtype: int64


### 4e.2 Run 2: new TPTP description

Based on the 55 TPTP bugs in the index, I rewrite only the TPTP line of the prompt. It names the common topics (profiling, tracing, tests, agents) and the documentation, readme and copyright bugs. It also adds a rule: if the text names TPTP, choose TPTP. The product name `Z_ARCHIVED` is not used, because it is a dataset artifact.

Nothing else changes (model, other components, input format). I run the same 16 reports and compare run 1 and run 2: severity, component, TPTP, and the other four components, to check that they did not get worse.

In [23]:
OLD_TPTP_LINE = "- TPTP: testing and profiling tools."
NEW_TPTP_LINE = (
    "- TPTP: the Eclipse Test and Performance Tools Platform. Profiling, tracing, test execution "
    "and test results, agents, and the documentation, readme, installation guide, copyright and "
    "maintenance bugs of TPTP itself. If the text names TPTP or a TPTP feature, choose TPTP, even "
    "when the bug is only about documentation, installation or copyright."
)
assert OLD_TPTP_LINE in TRIAGE_PROMPT_V1, "old TPTP line not found, check the exact text"
TRIAGE_PROMPT_V2 = TRIAGE_PROMPT_V1.replace(OLD_TPTP_LINE, NEW_TPTP_LINE)

In [25]:
subset = bugs_test_df[bugs_test_df["case_type"].isin(["sampled", "vague"])]
def run_triage_experiment(prompt, send_product, label):
    sev_ok, comp_ok, per_comp = 0, 0, {}
    print(f"=== {label} ===")
    for _, row in subset.iterrows():
        text = ""
        if send_product:
            text += f"Product: {row['product_name']}\n"
        text += (f"Title: {row['short_description']}\n"
                 f"Description: {str(row['long_description'])[:1500]}")
        try:
            t = triage_llm.invoke([("system", prompt), ("human", text)])
        except Exception as e:
            print(row["case_type"], "| ERROR:", e); continue
        print(row["case_type"], "| true:", row["severity"], row["component_name"],
              "| pred:", t.severity, t.component, "| conf:", t.confidence)
        if row["case_type"] == "sampled":
            sev_ok += t.severity == row["severity"]
            hit = t.component == row["component_name"]
            comp_ok += hit
            per_comp.setdefault(row["component_name"], []).append(hit)
    print("severity:", sev_ok, "/ 15 | component:", comp_ok, "/ 15")
    print({k: f"{sum(v)}/{len(v)}" for k, v in per_comp.items()}, "\n")

run_triage_experiment(TRIAGE_PROMPT_V1, True,  "Run 1: V1 prompt, with product")
run_triage_experiment(TRIAGE_PROMPT_V2, True,  "Run 2: V2 prompt, with product")
run_triage_experiment(TRIAGE_PROMPT_V2, False, "Run 3: V2 prompt, no product")

=== Run 1: V1 prompt, with product ===
sampled | true: high Core | pred: normal Releng | conf: 0.7
sampled | true: high UI | pred: high UI | conf: 0.85
sampled | true: high SWT | pred: normal SWT | conf: 0.85
sampled | true: high Core | pred: high Core | conf: 0.9
sampled | true: high TPTP | pred: low UI | conf: 0.75
sampled | true: low Releng | pred: low Releng | conf: 0.85
sampled | true: low UI | pred: low UI | conf: 0.85
sampled | true: low TPTP | pred: low UI | conf: 0.85
sampled | true: low UI | pred: low UI | conf: 0.85
sampled | true: low Releng | pred: low Releng | conf: 0.85
sampled | true: normal UI | pred: high Core | conf: 0.75
sampled | true: normal TPTP | pred: normal Releng | conf: 0.75
sampled | true: normal SWT | pred: normal SWT | conf: 0.85
sampled | true: normal TPTP | pred: low UI | conf: 0.7
sampled | true: normal TPTP | pred: low Other | conf: 0.6
vague | true:   | pred: normal Other | conf: 0.3
severity: 9 / 15 | component: 8 / 15
{'Core': '1/2', 'UI': '3/4', '

### 4e.3 Before and after

| | Run 1 (V1, with product) | Run 2 (V2, with product) | Run 3 (V2, no product) |
|---|---|---|---|
| Component correct | 8 / 15 | 12 / 15 | 14 / 15 |
| TPTP correct | 0 / 5 | 4 / 5 | 5 / 5 |
| Severity correct | 9 / 15 | 9 / 15 | 9 / 15 |

Only the TPTP line of the prompt changed between run 1 and run 2. The TPTP gain explains all 4 extra correct components. Severity did not change, because the fix did not target it. One TPTP report was still wrong in run 2. Run 3 is explained in sections 4e.4 and 4e.5.

The three runs were executed twice (once while building, once with "Restart and Run All" through the runner function in 4e.0). Both executions gave identical predictions and confidence values.

Limits: the failure was found on the test reports, although the new text came from the 55 TPTP bugs in the index. The sample is small (5 TPTP reports). The product name `Z_ARCHIVED` appears only in TPTP bugs, so it is a dataset shortcut. It is not used in the prompt.

### 4e.4 Run 3: shortcut check (no product name)

All 5 TPTP test reports have the product name `Z_ARCHIVED`, and no other test report has it. The model could use it as a hint. To check this, I run the same 16 reports with the `Product:` line removed from the input. The prompt is the same as run 2.

If TPTP stays near 4 / 5, the gain comes from the new prompt text. If it drops, the product name was helping. Either result is reported.

### 4e.5 Run 3 result and decision

Without the product name, the same 16 reports gave: component 14 / 15 (TPTP 5 / 5, Core 2 / 2, UI 3 / 4, SWT 2 / 2, Releng 2 / 2) and severity 9 / 15 (same as runs 1 and 2).

The TPTP gain therefore comes from the new prompt text, not from the product name `Z_ARCHIVED`. One Core report changed between run 2 and run 3. With one run per setting, I cannot say if this is the product name or random variation.

**Decision:** the final Triage node does not use the `Product:` line. The system then does not depend on a dataset artifact.

Limits: small sample, one run per setting, and the failure was found on the test reports. Severity (9 / 15) was not improved. One UI report is still predicted as Core in all three runs.

In [27]:
def triage_node(state: TriageState) -> dict:
    report = state["report"]
    text = (
        f"Title: {report['short_description']}\n"
        f"Description: {str(report.get('long_description', ''))[:1500]}"
    )
    try:
        result = triage_llm.invoke([("system", TRIAGE_PROMPT_V2), ("human", text)])
        update = {"triage": result.model_dump()}
        entry = (f"triage -> severity={result.severity} component={result.component} "
                 f"confidence={result.confidence}: {result.reason}")
    except Exception as e:
        update = {
            "triage": None,
            "final_decision": "needs_human_review",
            "review_reason": f"triage error: {e}",
        }
        entry = f"triage ERROR: {e}"
    update["step_count"] = state["step_count"] + 1
    update["log"] = state["log"] + [entry]
    return update

In [28]:
row = bugs_test_df[bugs_test_df["case_type"] == "vague"].iloc[0]
state = {"report": row.to_dict(), "router": None, "candidates": [], "duplicate": None,
         "triage": None, "final_decision": "", "review_reason": "", "step_count": 0, "log": []}
print(triage_node(state))

{'triage': {'severity': 'normal', 'component': 'Other', 'confidence': 0.3, 'reason': 'The report is extremely vague with no specific details about what "does not work" means (crash, hang, feature broken?), what the last update was, error messages, or steps to reproduce. Cannot determine the affected component without more information. Assigned to "Other" as a placeholder pending clarification.'}, 'step_count': 1, 'log': ['triage -> severity=normal component=Other confidence=0.3: The report is extremely vague with no specific details about what "does not work" means (crash, hang, feature broken?), what the last update was, error messages, or steps to reproduce. Cannot determine the affected component without more information. Assigned to "Other" as a placeholder pending clarification.']}
